In [9]:
# Import required packages
using DifferentialEquations     # For solving differential equations
using LinearAlgebra             # Provides linear algebra functionalities
using SparseArrays              # For efficient storage of sparse matrices
#using KLU                       # Sparse LU factorization solver
using IterativeSolvers          # Iterative algorithms for linear systems
using SparseDiffTools           # Tools for differentiating sparse functions
using Plots                     # For plotting results
using YAML                      # For parsing YAML files

# Ensure all packages are installed. If not, instruct the user to install them.
# Example: Pkg.add("DifferentialEquations"), etc.

include("Chemistry.jl") # Chemistry.jl is the actual program

atom_counter (generic function with 1 method)

In [10]:
# Set up the ODE problem
const FILENAME = "2step.yaml"

# Load the mechanism file
mechanism_data = load_mechanism_data(FILENAME)

# Process the chemical data
species_list, reaction_list, species_index_map = process_chemical_data(mechanism_data)

# Number of species and reactions
const n_species = length(species_list)
const n_vars = n_species + 1
const n_reactions = length(reaction_list)

# Build the stoichiometric matrix and kinetics list
S = build_stoichiometric_matrix(reaction_list, n_species)
kinetics_list = build_kinetics_list(reaction_list)

SplitKinetics(ElementaryKinetics[ElementaryKinetics(2.0e12, 0.0, 8365.4, false, false, Tuple{Int64, Float64}[], [3, 1], [1.0, 1.5], [4, 2], [1.0, 2.0]), ElementaryKinetics(6.324555e7, 0.0, 3346.1, false, true, Tuple{Int64, Float64}[], [4, 1], [1.0, 0.5], [5], [1.0])], [1, 2], FalloffKinetics[], Int64[])

In [11]:
config = ChemistryConfig(
        temperature = 1200.0, # Initial temperature in K
        pressure = 1e6,      # Initial pressure in Pa
        fuel_mixture = Dict("CH4" => 1/10.5), air_percentage = 9.5/10.5
)

X0 = initialize_concentrations(config, species_list, species_index_map)

7-element Vector{Float64}:
 1200.0
   19.16224093046556
    0.0
    9.62989588790502
    0.0
    0.0
   71.43437509856167

In [12]:
function dT(X::Vector{Float64}, r::Vector{Float64}, S::Array{Float64,2}, species_list::Vector{Species})
    n_species = length(species_list)
    n_reactions = length(r)

    h0_vec = zeros(n_reactions)      # Enthalpy change per reaction (J/mol)
    cp_vec = zeros(n_species)        # Heat capacity per species (J/(mol·K))

    T = X[1]                         # Temperature in K
    concentrations = X[2:end]        # Species concentrations in mol/m³

    # Pre-compute species enthalpies and heat capacities
    h_species = zeros(n_species)     # Enthalpy for each species (J/mol)
    for (species_index, species) in enumerate(species_list)
        cp_vec[species_index] = species_cp(T, species.thermo)
        h_species[species_index] = h0(T, species.thermo)
    end

    # Compute h0_vec for reactions
    for reaction_index in 1:n_reactions
        # Sum over species: stoichiometric coefficient * species enthalpy
        for species_index in 1:n_species
            stoich_coeff = S[species_index, reaction_index]
            if stoich_coeff != 0.0
                h0_vec[reaction_index] += stoich_coeff * h_species[species_index]
            end
        end
    end

    # Compute the total enthalpy change rate (J/(m³·s))
    dH = sum(r .* h0_vec)

    # Compute the total heat capacity of the mixture (J/(m³·K))
    c_p = sum(concentrations .* cp_vec)

    # Compute temperature rate of change (K/s)
    dT_dt = -dH / c_p

    return dT_dt
end
    
# Define the ODE function
function reaction_ode!(dX, X, p, t)
    # Unpack parameters
    S, kinetics_tuples, species_list, reaction_list, species_index_map = p

    X = max.(X,1e-50)

    r = zeros(length(reaction_list))

    # Compute reaction rates
    compute_reaction_rates!(r, X, kinetics_tuples, species_list)

    # Compute temperature rate of change
    dT_dt = dT(X, r, S, species_list)

    # Compute species concentration rate of change
    dC_dt = S * r  # dC/dt = S * r

    # Populate the derivative vector
    dX[1] = dT_dt       # Temperature derivative
    @views dX[2:end] .= dC_dt  # Species concentration derivatives
end

reaction_ode! (generic function with 1 method)

In [13]:
# Define time span and solver settings
tspan = (0.0, 3)
abstol, reltol = 1e-9, 1e-6

# Set up the ODE problem
params = S, kinetics_list, species_list, reaction_list, species_index_map
problem = ODEProblem(reaction_ode!, X0, tspan, params)

# Solve the ODE problem
@time sol = solve(problem,
            KenCarp4(autodiff = AutoFiniteDiff()),
            verbose=false,
            abstol=abstol,
            reltol=reltol)

  0.915319 seconds (1.19 M allocations: 60.866 MiB, 99.58% compilation time: 94% of which was recompilation)


retcode: Success
Interpolation: 3rd order Hermite
t: 33-element Vector{Float64}:
 0.0
 1.8480872919585158e-8
 2.0328960211543672e-7
 2.051376894073952e-6
 9.893771666799824e-6
 2.624916066985858e-5
 5.23199952950788e-5
 6.668873213058934e-5
 8.168723842232886e-5
 9.397434769312367e-5
 0.0001201047886529854
 0.00013995635579942958
 0.0001908563612025638
 ⋮
 0.002835797393805012
 0.0035307946383487197
 0.004511012490521877
 0.0053722038361745505
 0.006310744115200959
 0.00697120893963228
 0.008630352651452185
 0.012212493250484627
 0.03308996271425111
 0.24186465735191592
 2.3296116037285644
 3.0
u: 33-element Vector{Vector{Float64}}:
 [1200.0, 19.16224093046556, 0.0, 9.62989588790502, 0.0, 0.0, 71.43437509856167]
 [1200.1211634616086, 19.160898961061264, 0.0017889176901018857, 9.62900142905997, 0.0008938965716049305, 5.622734460122887e-7, 71.43437509856167]
 [1201.33825798899, 19.1474368745146, 0.019693518368516672, 9.620049128720764, 0.009778924835102652, 6.783434915568401e-5, 71.43437

In [14]:
# Extract the solution arrays
t = sol.t
T = sol[1, :]                  # Temperature over time
concentrations = sol[2:end, :]'  # Species concentrations over time

println(maximum(T))

# Plot results
plot1 = plot(t, T,
         xlabel = "Time (s)",
         ylabel = "Temperature (K)",
         legend = false)

plot2 = plot(t, concentrations,
         xlabel = "Time (s)",
         ylabel = "Concentration (mol/m³)",
         legend = false)

plot(plot1, plot2, layout = (2,1))
savefig("SingleNodeH2")

2483.3524620099606


"C:\\Users\\jelte\\chemicalcombustion\\SingleNodeH2.png"

In [15]:
using LinearAlgebra: norm

function compute_relaxation_time(sol; tol=1e-6)
    """
    Compute relaxation time τ for a solution `sol` of an ODEProblem.
    
    τ is defined as the time when ‖u(t) - u_steady‖ / ‖u0 - u_steady‖ ≈ 1/e.
    
    Args:
        sol: Solution object from DifferentialEquations.jl
        tol: Tolerance for checking convergence (default: 1e-6)
    
    Returns:
        τ: Relaxation time (first time when decay reaches 1/e)
        If no such time is found, returns `nothing`.
    """
    u0 = sol.prob.u0          # Initial (perturbed) state
    u_steady = sol[end]       # Equilibrium state (final value)
    Δ0 = norm(u0 - u_steady)  # Initial deviation magnitude
    
    # Handle cases where Δ0 ≈ 0 (no perturbation)
    if Δ0 < tol
        @warn "Initial state is already at equilibrium (‖Δu‖ = $Δ0). τ is undefined."
        return nothing
    end
    
    # Target deviation: Δ0 / e
    target_deviation = Δ0 / MathConstants.e
    
    # Find the first time when ‖u(t) - u_steady‖ ≤ target_deviation
    τ = nothing
    for (i, t) in enumerate(sol.t)
        Δu = norm(sol.u[i] - u_steady)
        if Δu ≤ target_deviation + tol  # Allow numerical tolerance
            τ = t
            break
        end
    end
    
    if τ === nothing
        @warn "No relaxation time found within solution timeframe. Increase `tspan`."
    end
    
    return τ
end

compute_relaxation_time(sol)

8.168723842232886e-5